In [1]:
import pandas as pd
import numpy as np

all cell types


In [5]:
df = pd.read_csv("/home/user-kp/anugreha/human/human protein atlas/rna_single_cell_type.tsv", sep="\t")
pivot_df = df.pivot(index="Gene name", columns="Cell type", values="nTPM").reset_index()
pivot_df.to_csv("/home/user-kp/anugreha/human/hpa_analysis/expression_all.csv", index=False)

In [6]:
# Create multiplication matrix
df = pd.read_csv("/home/user-kp/anugreha/human/hpa_analysis/expression_all.csv")
df.iloc[:, 1:] = (df.iloc[:, 1:] > 0).astype(int)
gene_matrix = pd.DataFrame(0, index=df["Gene name"], columns=df["Gene name"])

for i, gene1 in enumerate(df["Gene name"]):
    for j, gene2 in enumerate(df["Gene name"]):
        both_unexpressed = ((df.iloc[i, 1:] == 0) & (df.iloc[j, 1:] == 0)).sum()
        if both_unexpressed > 0:
            gene_matrix.loc[gene1, gene2] = 1
        else:
            gene_matrix.loc[gene1, gene2] = 0
gene_matrix.to_csv("/home/user-kp/anugreha/human/hpa_analysis/multiplication_matrix_HPA.csv")

In [2]:
def count_zero_interactions_fast(input_file):
    print("Loading interaction matrix...")
    df = pd.read_csv(input_file, index_col=0)
    matrix = df.values
    
    upper_triangle = np.triu(np.ones(matrix.shape), k=1).astype(bool)
    zero_count = np.sum(matrix[upper_triangle] == 0)
    total_pairs = np.sum(upper_triangle)
    
    print(f"\nAnalysis Complete:")
    print(f"Matrix shape: {matrix.shape}")
    print(f"Number of unique gene pairs with zero interaction: {zero_count:,}")
    print(f"Total number of unique gene pairs: {total_pairs:,}")
    print(f"Percentage of zero interactions: {(zero_count/total_pairs)*100:.2f}%")

input_file = "/home/user-kp/anugreha/human/hpa_analysis/multiplication_matrix_HPA.csv"
count_zero_interactions_fast(input_file)

Loading interaction matrix...

Analysis Complete:
Matrix shape: (12946, 12946)
Number of unique gene pairs with zero interaction: 56,452,996
Total number of unique gene pairs: 83,792,985
Percentage of zero interactions: 67.37%


downregulated genes (nTPM=0)

In [2]:
df = pd.read_csv("/home/user-kp/anugreha/human/human protein atlas/rna_celline.tsv", sep="\t")
k562_genes = df[(df["Cell line"] == "RPMI-8226") & (df["nTPM"] == 0)]["Gene name"].unique()
filtered_df = pd.DataFrame(k562_genes, columns=["Gene name"])
filtered_df.to_csv("/home/user-kp/anugreha/human/TS_plasma_cell/RPMI-8226_downreg.csv", index=False)

get counts for downregulated genes

In [ ]:
def extract_genes(filtered_csv, cancer_genes, output_csv):
    filtered_df = pd.read_csv(filtered_csv, index_col=0)
    cancer_gene_ids = pd.read_csv(cancer_genes)['Gene name'].str.strip().str.upper().drop_duplicates().tolist()

    print("Filtered DataFrame head:")
    print(filtered_df.head())
    print("Cancer gene IDs:")
    print(cancer_gene_ids[:10])
    print("Common genes:")
    common_genes = filtered_df.index[filtered_df.index.str.strip().str.upper().isin(cancer_gene_ids)]
    print(common_genes)

    extracted_df = filtered_df.loc[filtered_df.index.str.strip().str.upper().isin(cancer_gene_ids)]
    extracted_df.to_csv(output_csv)
    print(f"CSV file saved as {output_csv}")
filtered_csv = "/home/user-kp/anugreha/human/human protein atlas/matrix_erythroid.csv"
cancer_genes = "/home/user-kp/anugreha/human/hpa_analysis/k562_downreg.csv"
output_csv = "/home/user-kp/anugreha/human/hpa_2/k562_downreg_counts.csv"

extract_genes(filtered_csv, cancer_genes, output_csv)

Filtered DataFrame head:
        A1BG  A1CF  A2ML1  A4GALT  A4GNT  AAAS  AACS  AADAC  AADAT  AAGAB  \
Gene                                                                        
A1BG       0     0      0       0      0     0     0      0      0      0   
A1CF       0     1      1       0      1     0     0      0      0      0   
A2ML1      0     1      1       0      1     0     0      0      0      0   
A4GALT     0     0      0       0      0     0     0      0      0      0   
A4GNT      0     1      1       0      1     0     0      0      0      0   

        ...  ZSWIM5  ZSWIM6  ZUP1  ZW10  ZWINT  ZXDC  ZYG11B  ZYX  ZZEF1  ZZZ3  
Gene    ...                                                                     
A1BG    ...       0       0     0     0      0     0       0    0      0     0  
A1CF    ...       1       0     0     0      0     0       0    0      0     0  
A2ML1   ...       1       0     0     0      0     0       0    0      0     0  
A4GALT  ...       0       0   

In [2]:
def analysis(matrix_csv):
    try:
        df = pd.read_csv(matrix_csv, index_col=0)
        num_rows, num_cols = df.shape
        print(f"Shape of the matrix is: {num_rows} x {num_cols}")
       
        self_interactions = pd.DataFrame(False, index=df.index, columns=df.columns)
        for gene in df.index:
            if gene in df.columns:
                self_interactions.loc[gene, gene] = True
        num_self_interactions = self_interactions.sum().sum()
        print(f"\nNumber of possible self-interactions found: {num_self_interactions}")

        total_pairs = (num_rows * num_cols) - num_self_interactions
        print(f"Total number of possible interactions (excluding self-interactions): {total_pairs}")

        zero_counts = ((df == 0) & ~self_interactions).sum().sum()
        zero_percentage = (zero_counts / total_pairs) * 100
        
        print(f"Total zero interaction counts (excluding self-interactions): {zero_counts}")
        print(f"Percentage of zero counts (excluding self-interactions): {zero_percentage:.2f}%")
        
        non_zero_counts = total_pairs - zero_counts
        print(f"Total non-zero interaction counts: {non_zero_counts}")
        print(f"Percentage of non-zero counts: {100 - zero_percentage:.2f}%")
        
        zero_gene_rows = ((df == 0) & ~self_interactions).sum(axis=1).sort_values(ascending=False)
        top_10_rows = zero_gene_rows.head(10)
        print("\nRow genes with highest zero counts (excluding self-interactions):")
        print(top_10_rows)
   
        results = {
            'matrix_shape': (num_rows, num_cols),
            'self_interactions': num_self_interactions,
            'total_pairs_no_self': total_pairs,
            'zero_counts_no_self': zero_counts,
            'non_zero_counts': non_zero_counts,
            'zero_percentage_no_self': zero_percentage,
            'top_zero_genes_rows': top_10_rows
        }
        
        return results
        
    except FileNotFoundError:
        print(f"Error: The file {matrix_csv} was not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        return None

if __name__ == "__main__":
    matrix_csv = "/home/user-kp/anugreha/human/hpa_2/k562_downreg_counts.csv"
    analysis_results = analysis(matrix_csv)

Shape of the matrix is: 2239 x 12946

Number of possible self-interactions found: 2239
Total number of possible interactions (excluding self-interactions): 28983855
Total zero interaction counts (excluding self-interactions): 22723975
Percentage of zero counts (excluding self-interactions): 78.40%
Total non-zero interaction counts: 6259880
Percentage of non-zero counts: 21.60%

Row genes with highest zero counts (excluding self-interactions):
Gene
ZNF831     12945
ZNF704     12945
ZNF462     12945
AADAC      12945
ZAN        12945
XPNPEP2    12945
XIRP2      12945
XCL2       12945
XCL1       12945
WNT10A     12945
dtype: int64


druggable genes matrix

In [30]:
interaction_matrix = pd.read_csv("/home/user-kp/anugreha/human/hpa_2/k562_downreg_counts.csv")
druggable_genes = pd.read_csv("/home/user-kp/anugreha/human/DGIdb/druggable_genes.csv")["Gene"].tolist()

gene_headers = interaction_matrix.columns[1:]
druggable_gene_columns = [gene for gene in gene_headers if gene in druggable_genes or gene in interaction_matrix.iloc[:, 0].tolist()]

output_data = {"Downregulated Gene": interaction_matrix.iloc[:, 0]}
sl_columns = []

for idx, row in interaction_matrix.iterrows():
    sl_genes = []
    for gene in druggable_gene_columns:
        if row[gene] == 0:
            sl_genes.append(gene)
    
    for i, gene in enumerate(sl_genes):
        col_name = f"SL{i+1}"
        if col_name not in output_data:
            output_data[col_name] = [None] * len(interaction_matrix)
        sl_columns.append(col_name)
        output_data[col_name][idx] = gene

for col in sl_columns:
    if col not in output_data:
        output_data[col] = [None] * len(interaction_matrix)

output_df = pd.DataFrame(output_data)
output_df.to_csv("/home/user-kp/anugreha/human/hpa_2/k562_druggable.csv", index=False)

In [32]:
#druggable genes expression
def filter_druggable_gene_pairs(matrix_csv, druggable_genes_csv, output_csv):
    df = pd.read_csv(matrix_csv, index_col=0)
    druggable_genes = set(pd.read_csv(druggable_genes_csv)['Gene'].tolist())
    druggable_genes_in_matrix = druggable_genes.intersection(df.columns)
    
    output_data = []
    
    for downregulated_gene in df.index:
        interaction_scores = []
        for druggable_gene in druggable_genes_in_matrix:
            interaction_score = df.loc[downregulated_gene, druggable_gene]
            interaction_scores.append(interaction_score)
        
        if interaction_scores:
            output_data.append([downregulated_gene] + interaction_scores)
    
    output_df = pd.DataFrame(output_data, columns=['Downregulated Gene'] + list(druggable_genes_in_matrix))
    output_df.to_csv(output_csv, index=False)
    
    print(f"Filtered gene pairs saved as {output_csv}")

matrix_csv = "/home/user-kp/anugreha/human/hpa_2/k562_downreg_counts.csv"
druggable_genes_csv = "/home/user-kp/anugreha/human/DGIdb/druggable_genes.csv"
output_csv = "/home/user-kp/anugreha/human/hpa_2/k562_druggable_expression.csv"

filter_druggable_gene_pairs(matrix_csv, druggable_genes_csv, output_csv)


Filtered gene pairs saved as /home/user-kp/anugreha/human/hpa_2/k562_druggable_expression.csv


In [34]:
def process_gene_interaction(input_file_path, output_file_path):
    df = pd.read_csv(input_file_path)
    
    gene_a = []
    gene_b = []
    
    for index, row in df.iterrows():
        downregulated_gene = row.iloc[0]
        
        for col in df.columns[1:]:
            if row[col] == 0 and downregulated_gene != col:
                gene_a.append(downregulated_gene)
                gene_b.append(col)
    
    final_df = pd.DataFrame({"Gene A": gene_a, "Gene B": gene_b})
    
    final_df.to_csv(output_file_path, index=False)
    print(f"Processed data saved to {output_file_path}")

input_file_path = "/home/user-kp/anugreha/human/hpa_2/k562_druggable_expression.csv"
output_file_path = "/home/user-kp/anugreha/human/hpa_2/k562_SL_table.csv"
process_gene_interaction(input_file_path, output_file_path)

Processed data saved to /home/user-kp/anugreha/human/hpa_2/k562_SL_table.csv


GRN analysis

In [35]:
import networkx as nx

In [36]:
network_file = "/home/user-kp/anugreha/human/GRN/BioGRID_human_interactions.txt"
df = pd.read_csv(network_file,sep='\t',header =0)
G = nx.Graph()
G.add_edges_from(df.values)
genes_present = set(G.nodes)
print(df.head())

   Gene A  Gene B
0     BCR   HOXA9
1     ATM    TP53
2   NCOR1      AR
3  CTNNB1  CREBBP
4   BRCA1   CREB1


In [37]:
SL_pairs_file = "/home/user-kp/anugreha/human/hpa_2/k562_SL_table.csv"
sl_df = pd.read_csv(SL_pairs_file, sep = ',', header=0)
print(sl_df.head())
print(f"loaded {len(sl_df)} SL pairs")

  Gene A Gene B
0   A1CF   CTBS
1   A1CF  SPIDR
2   A1CF  PRDX5
3   A1CF   CBX1
4   A1CF  ALAS2
loaded 5941346 SL pairs


In [38]:
def compute_path(G, gene1, gene2):
    if gene1 not in genes_present or gene2 not in genes_present:
        return -2
    try:
        return nx.shortest_path_length(G,source=gene1,target=gene2)
    except nx.NetworkXNoPath:
        return -1
    
sl_df["Network_Distance"] = sl_df.apply(lambda row: compute_path(G, row["Gene A"], row["Gene B"]),axis=1)
sl_df.to_csv("/home/user-kp/anugreha/human/hpa_2/k562_SL_network_distance.csv",sep=',',index=False)
print("path calculated and saved!")

path calculated and saved!


In [39]:
valid_SL_df = sl_df[sl_df["Network_Distance"]==1]
valid_SL_df = valid_SL_df.sort_values(by="Network_Distance",ascending=True)
valid_SL_df.to_csv("/home/user-kp/anugreha/human/hpa_2/k562_SL_network_distance_filtered.csv",sep=',',index=False)
print("saved")

saved


In [40]:
count = (sl_df["Network_Distance"]==1).sum()
print(count)

796


remove predicted SL pairs

In [41]:
def filter_sl_pairs(network_file, predicted_sl_file, output_file):
    network_df = pd.read_csv(network_file)
    predicted_sl_df = pd.read_csv(predicted_sl_file)
    
    predicted_pairs = set()
    for _, row in predicted_sl_df.iterrows():
        gene_a, gene_b = row['Gene A'], row['Gene B']
        pair = tuple(sorted([gene_a, gene_b]))
        predicted_pairs.add(pair)
    
    removed_pairs = []
    filtered_rows = []
    
    for _, row in network_df.iterrows():
        gene_a, gene_b = row['Gene A'], row['Gene B']
        pair = tuple(sorted([gene_a, gene_b]))
        
        if pair in predicted_pairs:
            removed_pairs.append((gene_a, gene_b))
        else:
            filtered_rows.append(row)
    
    filtered_df = pd.DataFrame(filtered_rows)
    filtered_df.to_csv(output_file, index=False)
    
    print(f"Removed {len(removed_pairs)} pairs:")
    for pair in removed_pairs:
        print(f"- {pair[0]}, {pair[1]}")
    
    print(f"Filtered data saved to {output_file}")
    print(f"Original pairs: {len(network_df)}, Remaining pairs: {len(filtered_df)}")

network_file = "/home/user-kp/anugreha/human/hpa_2/k562_SL_network_distance_filtered.csv"
predicted_sl_file = "/home/user-kp/anugreha/human/SL_predictions/SL_predictions_merged.csv"
output_file = "/home/user-kp/anugreha/human/hpa_2/k562_SL_novel.csv"
filter_sl_pairs(network_file, predicted_sl_file, output_file)

Removed 343 pairs:
- UMOD, TP53
- ULK2, ULK3
- TYR, KRAS
- TTBK1, CSNK1D
- TRPC6, CSK
- TRAT1, FBXW7
- ATP2C2, ATP2C1
- ATP1A4, CSK
- APOBEC1, CSK
- ANGPTL3, CSK
- AMOT, CSK
- ALK, CDK9
- ADCYAP1R1, CSK
- ADCY8, KRAS
- ADAMTS12, ADAMTS10
- USP50, KRAS
- CCDC83, CSK
- CASP5, CSK
- CAPN9, CAPN2
- CAMK4, CAMK2G
- CAMK2A, DMPK
- CAMK1G, CAMK2G
- CAMK1G, MYC
- CAMK1G, PIK3CA
- CMYA5, CSK
- CLEC1A, CSK
- CHRNB3, CSK
- CHIA, CSK
- CFTR, CSK
- CDH19, FBXW7
- CDH19, CSK
- CDC42EP5, PTEN
- DBH, PIK3CD
- DAO, CSK
- DAGLA, DAGLB
- CYP2J2, CSK
- CYP24A1, CSK
- CYP11B1, KRAS
- CTTNBP2, KRAS
- COL12A1, FBXW7
- CNTN6, CSK
- CNTFR, RPS6KA4
- EGFR, XPO1
- ECEL1, ECE1
- EBF3, FBXW7
- DUSP2, DUSP6
- DSPP, CSK
- DQX1, CSK
- DNAAF1, CSK
- DISC1, CSK
- DDX60, CSK
- DCLK2, KRAS
- DBH, TK1
- EGFR, RPS2
- EGFR, CCND1
- EGFR, ILK
- EGFR, PPP2CA
- EGFR, RPL13A
- EGFR, RPL13
- EGFR, EEF1A1
- EGFR, HSPA8
- EGFR, PTEN
- EGFR, HSP90B1
- EGFR, BAP1
- EGFR, PARP2
- EGFR, CDK5
- EGFR, POLE3
- EGFR, ATP6V1B2
- EGFR, CDK9

comparison with experimental data

In [4]:
interactions_df = pd.read_csv('/home/user-kp/anugreha/human/k562_SL/gene_interactions.csv')
gene_pairs_df = pd.read_csv('/home/user-kp/anugreha/human/hpa_2/k562_SL_network_distance_filtered.csv')

interaction_pairs = set()
for _, row in interactions_df.iterrows():
    pair = tuple(sorted([row['Gene A'], row['Gene B']]))
    interaction_pairs.add((pair, row['Interaction Type']))

interaction_dict = {pair: interaction_type for pair, interaction_type in interaction_pairs}

results = []
for _, row in gene_pairs_df.iterrows():
    pair = tuple(sorted([row['Gene A'], row['Gene B']]))
    interaction_type = interaction_dict.get(pair, 'Unknown')
    results.append([row['Gene A'], row['Gene B'], interaction_type])

output_df = pd.DataFrame(results, columns=['Gene A', 'Gene B', 'Interaction Type'])
output_df.to_csv('/home/user-kp/anugreha/human/hpa_analysis/k562_comparison_all.csv', index=False)

checking expression of the druggable gene pair 